# XML Sales Data Warehouse
Run from this project directory after executing the SQL setup. See README.md for prerequisites and full-refresh behavior.

### 1. Data Ingestion 

In [ ]:
### 3 Files (XML) --> 3 DFs

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd

In [ ]:
tree = ET.parse("./Dataset/customers.xml")
root = tree.getroot()

lsit_customers = []

for customer in root.findall('customer'): 
    data  = {
        'customer_id' : customer.findtext('customer_id'),
        'first_name' : customer.findtext('first_name'),
        'last_name' : customer.findtext('last_name'),
        'email' : customer.findtext('email'),
        'city' : customer.findtext('city')
    }
    lsit_customers.append(data) 
df_customers = pd.DataFrame(lsit_customers)
df_customers.head(5)    

In [ ]:
tree = ET.parse("./Dataset/products.xml")
root = tree.getroot()

list_products = []

for product in root.findall('product'): 
    data  = {
        'product_id' : product.findtext('product_id'),
        'product_name' : product.findtext('product_name'),
        'category' : product.findtext('category'),
        'brand' : product.findtext('brand'),
        'price' : product.findtext('price')
    }
    list_products.append(data) 
df_products = pd.DataFrame(list_products)
df_products.head()   

In [ ]:
tree = ET.parse("./Dataset/orders.xml")
root = tree.getroot()

list_orders = []

for order in root.findall('order'): 
    data  = {
        'order_id' : order.findtext('order_id'),
        'customer_id' : order.findtext('customer_id'),
        'product_id' : order.findtext('product_id'),
        'quantity' : order.findtext('quantity'),
        'order_date' : order.findtext('order_date')
    }
    list_orders.append(data) 
df_orders = pd.DataFrame(list_orders)
df_orders.head()

In [ ]:
df_orders
df_customers
df_products

In [ ]:
df_orders.columns

### 2. Bronze Layer

In [ ]:
### 3 DFs --> SQL Server (3 Tables Bronze)

In [ ]:
### Pip install sqlalchemy

In [ ]:
from sqlalchemy import create_engine, text

engine = create_engine(
    "mssql+pyodbc://localhost/project1_dwh"
    "?trusted_connection=yes"
    "&driver=ODBC+Driver+17+for+SQL+Server"
    "&TrustServerCertificate=yes"
)

In [ ]:
with engine.begin() as connection:
    connection.execute(text("DELETE FROM bronze.customers"))
    df_customers.to_sql(
        'customers', 
        con = connection, 
        schema = 'bronze',
        index = False, 
        if_exists = 'append'
        )


In [ ]:
with engine.begin() as connection:
    connection.execute(text("DELETE FROM bronze.products"))
    df_products.to_sql(
        'products', 
        con = connection, 
        schema = 'bronze',
        index = False, 
        if_exists = 'append'
        )


In [ ]:
with engine.begin() as connection:
    connection.execute(text("DELETE FROM bronze.orders"))
    df_orders.to_sql(
        'orders', 
        con = connection, 
        schema = 'bronze',
        index = False, 
        if_exists = 'append'
        )


### 3. Silver Layer

In [ ]:
### SQL Server (3 Tables Bronze) --> SQL Server (3 Tables Silver)

In [ ]:
from sqlalchemy import create_engine, text

engine = create_engine(
    "mssql+pyodbc://localhost/project1_dwh"
    "?trusted_connection=yes"
    "&driver=ODBC+Driver+17+for+SQL+Server"
    "&TrustServerCertificate=yes"
)

In [ ]:
df_customers = pd.read_sql( 'SELECT * FROM bronze.customers', con = engine)
df_products = pd.read_sql( 'SELECT * FROM bronze.products', con = engine)
df_orders = pd.read_sql( 'SELECT * FROM bronze.orders', con = engine)

In [ ]:
df_customers.head()

In [ ]:
df_customers.dtypes

In [ ]:
df_customers['customer_id'] = df_customers['customer_id'].astype(int)

In [ ]:
df_customers.dtypes

In [ ]:
df_customers['full_name'] = df_customers['first_name'] + ' ' + df_customers['last_name']

In [ ]:
df_customers.drop(columns=['first_name', 'last_name'], inplace=True)

In [ ]:
df_customers.isnull().sum()

In [ ]:
df_products.dtypes

In [ ]:
df_products['product_id'] = df_products['product_id'].astype(int)
df_products['price'] = df_products['price'].astype(float)

In [ ]:
df_products

In [ ]:
df_orders.dtypes

In [ ]:
df_orders['order_id'] = df_orders['order_id'].astype(int)
df_orders['customer_id'] = df_orders['customer_id'].astype(int)
df_orders['product_id'] = df_orders['product_id'].astype(int)
df_orders['quantity'] = df_orders['quantity'].astype(int)
df_orders['order_date'] = pd.to_datetime(df_orders['order_date'])

In [ ]:
df_customers
df_products
df_orders

In [ ]:
with engine.begin() as connection:
    connection.execute(text("DELETE FROM silver.customers"))
    df_customers.to_sql(
        'customers', 
        con = connection, 
        schema = 'silver',
        index = False, 
        if_exists = 'append'
        )


In [ ]:
with engine.begin() as connection:
    connection.execute(text("DELETE FROM silver.products"))
    df_products.to_sql(
        'products', 
        con = connection, 
        schema = 'silver',
        index = False, 
        if_exists = 'append'
        )


In [ ]:
with engine.begin() as connection:
    connection.execute(text("DELETE FROM silver.orders"))
    df_orders.to_sql(
        'orders', 
        con = connection, 
        schema = 'silver',
        index = False, 
        if_exists = 'append'
        )


### 4. Gold Layer

In [ ]:
### SQL Server (3 Tables Silver) --> SQL Server (One Big Table Gold)

In [ ]:
from sqlalchemy import create_engine, text

engine = create_engine(
    "mssql+pyodbc://localhost/project1_dwh"
    "?trusted_connection=yes"
    "&driver=ODBC+Driver+17+for+SQL+Server"
    "&TrustServerCertificate=yes"
)

In [ ]:
df_customers = pd.read_sql( 'SELECT * FROM silver.customers', con = engine)
df_products = pd.read_sql( 'SELECT * FROM silver.products', con = engine)
df_orders = pd.read_sql( 'SELECT * FROM silver.orders', con = engine)

In [ ]:
df_output = pd.merge(df_orders, df_customers, how = 'inner', on = 'customer_id')
df_output = pd.merge(df_output, df_products, how = 'inner', on = 'product_id')
df_output.head()

In [ ]:
df_output['total_amount'] = df_output['quantity'] * df_output['price']

In [ ]:
df_output.head()

In [ ]:
df_output.columns

In [ ]:
with engine.begin() as connection:
    connection.execute(text("DELETE FROM gold.fact"))
    df_output.to_sql(
        'fact', 
        con = connection, 
        schema = 'gold',
        index = False, 
        if_exists = 'append'
        )
